In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/diabetic_data.csv', na_values=['?', 'Unknown/Invalid'])

print(f"Shape: {df.shape}")
print(f"\nColumns with missing values:")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    'missing_count': missing[missing > 0],
    'missing_pct': missing_pct[missing > 0]
}).sort_values('missing_pct', ascending=False)
print(missing_summary)

/tmp/ipykernel_2450808/205546032.py:4: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/diabetic_data.csv', na_values=['?', 'Unknown/Invalid'])


Shape: (101766, 50)

Columns with missing values:
                   missing_count  missing_pct
weight                     98569        96.86
max_glu_serum              96420        94.75
A1Cresult                  84748        83.28
medical_specialty          49949        49.08
payer_code                 40256        39.56
race                        2273         2.23
diag_3                      1423         1.40
diag_2                       358         0.35
diag_1                        21         0.02
gender                         3         0.00


## Missing Value Strategy

| Column | Missing % | Strategy | Rationale |
|---|---|---|---|
| weight | 96.86% | Convert to binary `weight_recorded` flag, drop original | Too sparse to model; recorded/not-recorded may itself be predictive |
| max_glu_serum | 94.75% | Treat missing as category "not_tested" | Lab not ordered = clinical judgment, that's informative |
| A1Cresult | 83.28% | Treat missing as category "not_tested" | Same reasoning as max_glu_serum |
| medical_specialty | 49.08% | Treat missing as category "unknown" | Administrative gap, keep signal in 51% recorded |
| payer_code | 39.56% | Treat missing as category "unknown" | Same — administrative |
| race | 2.23% | Treat missing as category "unknown" | Light enough to keep category |
| diag_1/2/3 | <1.5% | Drop affected rows | Light, clinically meaningful — better to drop than fake |
| gender | 0.003% | Drop affected rows | Trivial |

In [2]:
df.discharge_disposition_id.value_counts()

discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
8       108
15       63
24       48
9        21
17       14
16       11
19        8
10        6
27        5
12        3
20        2
Name: count, dtype: int64

In [3]:
# discharge_disposition_id values that mean the patient cannot be readmitted.
# These come from IDS_mapping.csv 
# Common values for expired/hospice in this dataset:
#   11: Expired
#   13: Hospice / home
#   14: Hospice / medical facility
#   19: Expired at home (medicaid)
#   20: Expired in medical facility (medicaid)
#   21: Expired, place unknown (medicaid)

unreachable_dispositions = [11, 13, 14, 19, 20, 21]

before = len(df)
df = df[~df['discharge_disposition_id'].isin(unreachable_dispositions)].copy()
after = len(df)

print(f"Dropped {before - after:,} unreachable encounters")
print(f"Remaining: {after:,}")

Dropped 2,423 unreachable encounters
Remaining: 99,343


In [4]:
# CReate binary target
# Binary target: 1 if readmitted within 30 days, 0 otherwise
df['target'] = (df['readmitted'] == '<30').astype(int)

# Drop the original readmitted column — it's now the target
df = df.drop(columns=['readmitted'])

# Verify
print(f"Target distribution:")
print(df['target'].value_counts())
print(f"\nPositive class rate: {df['target'].mean()*100:.2f}%")

Target distribution:
target
0    88029
1    11314
Name: count, dtype: int64

Positive class rate: 11.39%


In [5]:
# Apply missing-value strategies
# weight: convert to binary recorded-flag, drop original
df['weight_recorded'] = df['weight'].notna().astype(int)
df = df.drop(columns=['weight'])

# max_glu_serum, A1Cresult: treat missing as 'not_tested'
df['max_glu_serum'] = df['max_glu_serum'].fillna('not_tested')
df['A1Cresult'] = df['A1Cresult'].fillna('not_tested')

# medical_specialty, payer_code, race: treat missing as 'unknown'
df['medical_specialty'] = df['medical_specialty'].fillna('unknown')
df['payer_code'] = df['payer_code'].fillna('unknown')
df['race'] = df['race'].fillna('unknown')

# diag_1/2/3, gender: drop affected rows
before = len(df)
df = df.dropna(subset=['diag_1', 'diag_2', 'diag_3', 'gender'])
after = len(df)
print(f"Dropped {before - after:,} rows with missing diagnoses or gender")
print(f"Final cohort: {after:,} rows")

Dropped 1,521 rows with missing diagnoses or gender
Final cohort: 97,822 rows


In [6]:
# Confirm no missing values remain
remaining_missing = df.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
if len(remaining_missing) == 0:
    print("No missing values remain")
else:
    print("Missing values still present:")
    print(remaining_missing)

print(f"\nFinal shape: {df.shape}")
print(f"Target rate: {df['target'].mean()*100:.2f}%")

No missing values remain

Final shape: (97822, 50)
Target rate: 11.46%


In [8]:
# Save the cleaned analytical dataset
df.to_csv('../data/cleaned_diabetes.csv', index=False)
print(f"Saved cleaned dataset: {df.shape}")

Saved cleaned dataset: (97822, 50)
